In [4]:
import pandas as pd
import numpy as np

# -----------------------------
# Load datasets
# -----------------------------
philly_stops = pd.read_csv("philly_stops.csv", low_memory=False,on_bad_lines="skip",)
eagles = pd.read_csv("cleanedEaglesData.csv")

# -----------------------------
# Parse dates
# -----------------------------
philly_stops['date'] = pd.to_datetime(philly_stops['date'], format='mixed', errors='coerce').dt.date

eagles['Date'] = pd.to_datetime(eagles['Date']).dt.date

philly_stops = philly_stops.dropna(subset=['date'])

# -----------------------------
# Keep only needed columns
# -----------------------------
philly_stops = philly_stops[['date','time','lat','lng']]

# Remove NA rows
philly_stops = philly_stops.dropna()

# -----------------------------
# Calculate distance from stadium
# -----------------------------
stadium_lat = 39.9014
stadium_lng = -75.1676

lat1 = np.radians(philly_stops['lat'])
lon1 = np.radians(philly_stops['lng'])

lat2 = np.radians(stadium_lat)
lon2 = np.radians(stadium_lng)

dlat = lat1 - lat2
dlon = lon1 - lon2

a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
c = 2*np.arcsin(np.sqrt(a))

earth_radius_km = 6371
philly_stops['distance_km'] = earth_radius_km * c

# -----------------------------
# Create ring regions
# -----------------------------
philly_stops['region'] = np.ceil(philly_stops['distance_km']).astype('Int64')

# Keep only 5km around stadium
philly_stops.loc[philly_stops['region'] > 5, 'region'] = pd.NA

philly_stops.dropna(subset=['region'], inplace=True)

# Drop location columns
philly_stops.drop(columns=['lat','lng','distance_km'], inplace=True)

# -----------------------------
# Create hour column
# -----------------------------
philly_stops['hour'] = pd.to_datetime(
    philly_stops['time'],
    format='mixed',
    errors='coerce'
).dt.hour

philly_stops = philly_stops.dropna(subset=['hour'])

philly_stops.drop(columns=['time'], inplace=True)

# -----------------------------
# Aggregate stop counts
# -----------------------------
aggregate_philly_stops = (
    philly_stops
    .groupby(['date','hour','region'])
    .size()
    .reset_index(name='stop_count')
)

# -----------------------------
# Fill missing hour/region combos
# -----------------------------
full_index = pd.MultiIndex.from_product(
    [
        aggregate_philly_stops['date'].unique(),
        range(24),
        range(1,6)
    ],
    names=['date','hour','region']
)

aggregate_philly_stops = (
    aggregate_philly_stops
    .set_index(['date','hour','region'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# -----------------------------
# Merge Eagles game data
# -----------------------------
merged_df = aggregate_philly_stops.merge(
    eagles,
    how='left',
    left_on='date',
    right_on='Date'
)

merged_df['gameday'] = merged_df['Date'].notna().astype(int)

merged_df = merged_df.drop(columns=['Date'])

# -----------------------------
# Calculate baseline (non-game days)
# -----------------------------
non_gamedays = merged_df[merged_df['gameday'] == 0].copy()

baseline_stops = (
    non_gamedays
    .groupby(['hour','region'])['stop_count']
    .mean()
    .reset_index(name='baseline')
)

# -----------------------------
# Filter to game days
# -----------------------------
gamedays = merged_df[merged_df['gameday'] == 1].copy()

# -----------------------------
# Create start/end hours
# -----------------------------
gamedays['start_hour'] = (
    pd.to_datetime(gamedays['StartTime'], format='%H:%M:%S')
    .dt.round('h')
    .dt.hour
)

gamedays['end_hour'] = (
    pd.to_datetime(gamedays['xEndTime'], format='%H:%M:%S')
    .dt.round('h')
    .dt.hour
)

# -----------------------------
# Classify before/during/after
# -----------------------------
conditions = [
    gamedays['hour'] < gamedays['start_hour'],
    (gamedays['hour'] >= gamedays['start_hour']) &
    (gamedays['hour'] <= gamedays['end_hour']),
    gamedays['hour'] > gamedays['end_hour']
]

choices = ['before','during','after']

gamedays['game_period'] = np.select(conditions, choices, default="unknown")

# -----------------------------
# Add baseline
# -----------------------------
gamedays = gamedays.merge(
    baseline_stops,
    on=['hour','region'],
    how='left'
)

# -----------------------------
# Create percent change variable
# -----------------------------
gamedays['percent_change_in_stops'] = (
    (gamedays['stop_count'] - gamedays['baseline']) /
    gamedays['baseline']
)

# -----------------------------
# Final dataset
# -----------------------------
final_df = gamedays.copy()

final_df.drop(columns=['baseline'], inplace=True)

# -----------------------------
# Export
# -----------------------------
final_df.to_csv("eagles_modeling_df.csv", index=False)

print("Final dataset created:", final_df.shape)

Final dataset created: (8040, 34)


In [11]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# load final dataset you created earlier
df = pd.read_csv("eagles_modeling_df.csv")

print(df.head())
print(df.shape)

         date  hour  region  stop_count Game     Day StartTime  xEndTime  \
0  2014-09-07     0       1          13    1  Sunday  13:02:00  16:14:00   
1  2014-09-07     0       2          10    1  Sunday  13:02:00  16:14:00   
2  2014-09-07     0       3          27    1  Sunday  13:02:00  16:14:00   
3  2014-09-07     0       4          76    1  Sunday  13:02:00  16:14:00   
4  2014-09-07     0       5          61    1  Sunday  13:02:00  16:14:00   

   Home                   Opp  ...  Cowboys  RegularSeason  Division  \
0   0.0  Jacksonville Jaguars  ...      0.0            1.0       0.0   
1   0.0  Jacksonville Jaguars  ...      0.0            1.0       0.0   
2   0.0  Jacksonville Jaguars  ...      0.0            1.0       0.0   
3   0.0  Jacksonville Jaguars  ...      0.0            1.0       0.0   
4   0.0  Jacksonville Jaguars  ...      0.0            1.0       0.0   

   ConfChamp  SuperBowl  gameday  start_hour  end_hour  game_period  \
0        0.0        0.0        1       

In [24]:
df = df.dropna(subset=["percent_change_in_stops"])
df = pd.get_dummies(df, columns=["game_period"], drop_first=True)

In [25]:
features = [
    "hour",
    "region",
    "Home",
    "RegularSeason",
    "Division",
    "ConfChamp",
    "SuperBowl",
    "gameday",
    "start_hour",
    "end_hour"
]

target = "percent_change_in_stops"

X = df[features]
y = df[target]

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [27]:
model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [28]:
predictions = model.predict(X_test)

In [29]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

MAE: 1.4027935986525553
RMSE: 2.3271343300236262
R^2: 0.1327660459070299


In [30]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

print(coefficients.sort_values(by="Coefficient", ascending=False))

         Feature   Coefficient
3  RegularSeason  1.221100e+00
8     start_hour  2.457141e-02
9       end_hour  1.824761e-02
7        gameday  2.775558e-17
0           hour  0.000000e+00
5      ConfChamp -2.217224e-01
6      SuperBowl -2.259241e-01
1         region -6.139261e-01
4       Division -7.734539e-01
2           Home -1.268736e+00


On average, the model’s prediction is off by about 1.4 percentage points in the percent change in stops

The model explains about 13% of the variation in percent changes in police stops.

